In [111]:
import pandas as pd
from six_nations_fantasy.data import load_dataset
from six_nations_fantasy.models import PositionShrinkageMean

In [33]:
data = load_dataset()
pm = data.player_matches()
matches = data.matches
players = pd.read_parquet("../data/rebuild/players.parquet")

In [122]:
def predicted_skill_column_name(stat):
    return f"skill({stat})"

def skill(training, fixtures, stat):
    model = PositionShrinkageMean(k=20, scorer=lambda x: x[stat])
    model.fit(training)

    predicted = model.predict(fixtures).rename(predicted_skill_column_name(stat))
    baseline = training.performances[stat].mean()
    
    return 1 - predicted / baseline

In [123]:
start_date = pd.to_datetime("2026-02-05")
end_date = pd.to_datetime("2026-03-14")

training = data.select_matches(matches.loc[matches["date"] < start_date, "match_id"])
test = pm[(pm["date"].between(start_date, end_date, inclusive="both")) & (pm["competition_id"] == 0)]

In [95]:
# The m most skilled players (relative to their position) at the given stat 
# for the upcoming fixtures
def most_skilled(stat, m=5):
    ratings = skill(training, test, stat)
    
    return (pd.concat([ratings, test["player_id"]], axis=1)
        .groupby("player_id")
        .agg(predicted=(predicted_skill_column_name(stat), "mean"))
        .nsmallest(m, columns="predicted")
        .join(players["name"], on="player_id")
    )

In [128]:
# Now find the most skilled players in general (the players with the highest skill in any particular stat)
selected_stats = ['conversions', 'penalty_goals', 'drop_goals', 'carries', 'clean_breaks',
       'completed_tackles', 'turnovers_conceded', 'turnovers_won',
       'total_kicks', 'passes', 'carried_meters', 'offloads',
       'defenders_beaten', 'try_assists', 'tries', 'carries_per_minute',
       'total_tackles', 'dominant_tackles', 'ruck_turnovers',
       'lineouts_won', 'tackles_per_minute']

all_skills = pd.concat([skill(training, test, stat) for stat in selected_stats] + [test["player_id"]], axis=1)
best = (all_skills
    .drop_duplicates(subset=["player_id"])
    .set_index("player_id")
    .agg(["idxmax", "min"], axis=1)
    .astype({"min": "float64"})
    .nsmallest(10, columns="min")
    .join(players["name"], on="player_id")
)

In [129]:
best

,idxmax,min,name
player_id,,,
102642,skill(ruck_turnovers),-56.264957,George Ford
240455,skill(lineouts_won),-34.903590,Daniel Edwards
118619,skill(lineouts_won),-31.639627,Sam Costelow
175123,skill(tackles_per_minute),-27.389182,Thomas Ramos
137626,skill(penalty_goals),-26.869165,Finlay Smith
157863,skill(ruck_turnovers),-22.590129,Finn Russell
241698,skill(lineouts_won),-20.383548,Sam Prendergast
211240,skill(conversions),-17.873382,Dan Sheehan
190344,skill(lineouts_won),-17.375344,Matthieu Jalibert


The most "skilled" players are those that have been very good at rare stats for their position and haven't been shrunk enough (George Ford ruck turnovers, Dan Edwards lineouts won, etc.).

In [136]:
(all_skills
    .quantile(q=0.1, axis=0)
    .drop("player_id")
    .sort_values()
)

skill(total_kicks)          -2.458200
skill(clean_breaks)         -2.212850
skill(try_assists)          -2.094721
skill(ruck_turnovers)       -1.956970
skill(carried_meters)       -1.891685
skill(tries)                -1.612154
skill(defenders_beaten)     -1.563372
skill(passes)               -1.480067
skill(offloads)             -1.290724
skill(turnovers_conceded)   -1.200137
skill(turnovers_won)        -1.178857
skill(dominant_tackles)     -0.850123
skill(completed_tackles)    -0.844764
skill(carries)              -0.822641
skill(total_tackles)        -0.776189
skill(tackles_per_minute)   -0.390009
skill(carries_per_minute)   -0.336152
skill(conversions)           0.122334
skill(lineouts_won)          0.258100
skill(penalty_goals)         0.319249
skill(drop_goals)            0.754274
Name: 0.1, dtype: float64

This is more interesting: the skills with the best 90th percentile players are total_kicks, clean_breaks, try_assists, ruck_turnovers and carried_meters. This matches my expectation. Next down the list are tries, which I feel like is a natural point to cut off the list of skills we care about in this analysis.

In [137]:
most_skilled("total_kicks")

,predicted,name
player_id,,
177559,-8.376388,Alex Mitchell
137493,-8.166716,Antoine Dupont
190344,-7.666154,Matthieu Jalibert
133163,-7.443753,Tomos Williams
102642,-7.053742,George Ford


In [138]:
most_skilled("clean_breaks")

,predicted,name
player_id,,
110793,-4.295421,Cadan Murley
237223,-3.939398,Louis Bielle-Biarrey
156918,-3.878865,Tommy Freeman
109852,-3.771129,James Lowe
110170,-3.570720,Kyle Steyn


These lists pass the visual test for me, as these are players that are generally quite good in their positions (e.g. Murley, LBB and Freeman for clean_breaks).